# Applied Machine Learning (CSAI2017P) — Lab Assignment 4
## Customer Churn Prediction

Experiments 5 · CO2, CO4 · **10 marks**

| | |
|---|---|
| Name | Anirudh Awasthi |
| SAP ID | 590027460 |
| Batch | 74 |
| Due | announced in the lab |
| Lab quiz | LQ4, after submission |

**Datasets:** A2 (Telco Customer Churn)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


numpy        2.5.2
pandas       3.0.5
sklearn      1.9.0


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [1]:
import pandas as pd

df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# TotalCharges looks numeric but is stored as text
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Shape:", df.shape)
display(df.head())

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Encode and fit  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2. Target: `Churn`.*

a. Build a `ColumnTransformer` with one-hot encoding for categoricals and scaling for numerics.
b. Fit a logistic regression inside the pipeline and report test accuracy.
c. State the churn rate in the data and compare it with your accuracy in two lines.


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

RANDOM_STATE = 0
# Separate features and target
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Complete pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

# Fit model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Churn rate
churn_rate = y.mean()

print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Churn Rate: {churn_rate:.4f} ({churn_rate*100:.2f}%)")

C:\Users\Anirudh Awasthi\AppData\Local\Temp\ipykernel_31044\3727991350.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns


Test Accuracy: 0.8034 (80.34%)
Churn Rate: 0.2654 (26.54%)


**Observation (A1):**
The logistic regression model achieved a test accuracy of 80.34%, while the overall churn rate was 26.54%. The accuracy is much higher than the churn rate because accuracy also includes the majority non-churn class. Therefore, accuracy alone does not fully describe how well the model identifies customers who actually churn.

## A2. Confusion matrix  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Print the confusion matrix and the classification report for your A1 model.
b. State the number of churners the model missed.
c. In two lines, say which error costs the company more: a missed churner, or a retention offer sent to someone who was staying.


In [4]:
from sklearn.metrics import confusion_matrix, classification_report

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

# Number of actual churners missed by the model
missed_churners = cm[1, 0]

print(f"\nChurners missed by the model: {missed_churners}")


Confusion Matrix:
[[923 112]
 [165 209]]

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.85      0.89      0.87      1035
       Churn       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409


Churners missed by the model: 165


**Observation (A2):**
The confusion matrix shows that the model missed 165 actual churners, which are false negatives. For a telecom company, a missed churner is generally more costly because the company loses the opportunity to retain that customer. A retention offer sent to a customer who would have stayed may have some cost, but it is usually less costly than losing a customer.

## A3. Threshold moving  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Take `predict_proba` from your model and re-classify at thresholds 0.5, 0.35 and 0.25.
b. Report precision and recall on the churn class for each.
c. State which threshold you would ship and why, in two lines.


**Observation(A3):** 

---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Model comparison  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Train logistic regression, a decision tree (`max_depth=5`) and a random forest on the same pipeline.
b. Report accuracy, precision, recall, F1 and ROC-AUC for each in one table.
c. Name the best model for this problem and defend the choice in 4–6 lines, referring to the cost argument.


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE
    )
}

results = []

for name, classifier in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    pipeline.fit(X_train, y_train)

    y_pred_model = pipeline.predict(X_test)
    y_prob_model = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred_model),
        "Precision": precision_score(y_test, y_pred_model),
        "Recall": recall_score(y_test, y_pred_model),
        "F1": f1_score(y_test, y_pred_model),
        "ROC-AUC": roc_auc_score(y_test, y_prob_model)
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.8034,0.6511,0.5588,0.6014,0.8515
1,Decision Tree,0.7935,0.6095,0.6176,0.6135,0.8368
2,Random Forest,0.7864,0.6327,0.4652,0.5362,0.8256


**Observation (B1):**
Logistic Regression achieved the highest accuracy (80.34%) and ROC-AUC (0.8515), while Decision Tree had slightly higher recall (61.76%) and F1-score (0.6135). Random Forest performed lowest on recall and F1-score. Logistic Regression was selected as the best overall model due to its stronger ROC-AUC and accuracy.

## B2. Class imbalance  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Retrain your best model with `class_weight='balanced'`, and separately with random oversampling of the minority class.
b. Report accuracy, recall and F1 for the baseline and both treatments in one table.
c. Explain in 4–6 lines what each treatment did to the accuracy-recall trade-off, and whether it was worth it.


**Observation (B2):**


---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Which features carry the signal  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Report the ten most important features from your best model (coefficients or feature importances).
b. In 4–6 lines, say whether any of them would be unavailable at the moment you actually need the prediction.


In [8]:

best_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

best_model.fit(X_train, y_train)

# Get transformed feature names
feature_names = best_model.named_steps[
    "preprocessor"
].get_feature_names_out()

# Get Logistic Regression coefficients
coefficients = best_model.named_steps[
    "classifier"
].coef_[0]

# Create feature-importance table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute Coefficient": abs(coefficients)
})

top_10 = feature_importance.sort_values(
    "Absolute Coefficient",
    ascending=False
).head(10)

display(top_10)

,Feature,Coefficient,Absolute Coefficient
1,num__tenure,-1.389515,1.389515
38,cat__Contract_Two year,-0.767838,0.767838
3,num__TotalCharges,0.694822,0.694822
36,cat__Contract_Month-to-month,0.603141,0.603141
15,cat__InternetService_DSL,-0.364858,0.364858
11,cat__PhoneService_Yes,-0.334995,0.334995
20,cat__OnlineSecurity_Yes,-0.323951,0.323951
39,cat__PaperlessBilling_No,-0.318645,0.318645
29,cat__TechSupport_Yes,-0.277156,0.277156
12,cat__MultipleLines_No,-0.270950,0.270950


**Observation (C1):**
The strongest signals were tenure (-1.3895), two-year contract (-0.7678), and TotalCharges (0.6948). Month-to-month contracts also showed a strong positive coefficient (0.6031), indicating higher churn tendency. These features are generally available at the time of prediction, so they can be used for churn prediction.

---

## Self-check before submitting

- [ ] Kernel restarted and run top-to-bottom, all outputs visible
- [ ] Every attempted question has a written observation
- [ ] No transformer fitted outside a pipeline
- [ ] Metrics reported with units where they have any
- [ ] File named `AML_LA04_<SAPID>.ipynb`
